# Lab 2: Logistic Regression

CS 300 - Introduction to Machine Learning

Date - Nov 24, 2026


## 1. Data Preparation and Exploration

In [9]:
import pandas as pd

data_filepath = "../datasets_globelEnv/ML_parami_1.csv"
df = pd.read_csv(data_filepath)
display(df.head())

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [10]:
print("\n1. Data Preparation and Exploration")

# a. Display the number of observations and variables.
print(f"Shape of df: {df.shape}")
print()

# b. Identify missing values in each variable.
print(df.isnull().sum())
print()

# c. Identify duplicated observations.
print(f"Number of duplicated rows: {df.duplicated().sum()}")
print()

# d. Examine the distribution of the target variable TenYearCHD.
print("Target variable counts: ")
print(df['TenYearCHD'].value_counts())
print()

# e. Comment on whether the target variable is balanced or imbalanced.
print("Target variable percentages: ")
print((df["TenYearCHD"].value_counts(normalize=True)*100).round(2))


1. Data Preparation and Exploration
Shape of df: (4238, 16)

male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

Number of duplicated rows: 0

Target variable counts: 
TenYearCHD
0    3594
1     644
Name: count, dtype: int64

Target variable percentages: 
TenYearCHD
0    84.8
1    15.2
Name: proportion, dtype: float64


a. Observations and Variables: The dataset contains 4,238 observations (rows) and 16 variables (columns). i.e. 15 input features and 1 binary target variable (TenYearCHD).

b. Missing Values: Seven variables have missing values: glucose (388), education (105), BPMeds (53), totChol (50), cigsPerDay (29), BMI (19) and heartRate (1). The remaining 9 columns have 0 missing values.

c. Duplicated Observations: There are 0 duplicated row in the dataset.

d. Target Distribution (TenYearCHD):
|TenYearCHD|Total Patients|Percentage|
|---|---|---|
|0 (No CHD)| 3594 patients | 84.8% |
|1 (Yes CHD)| 644 patients | 15.2% |


e. Balanced or Imbalanced? The target variable is imbalanced with an approximate 85:15 ratio between negative (0) and positive (1) cases.

f. Predicting class 0 for everyone would give 84.80% accuracy on the full dataset while detecting zero CHD cases. Precision, recall and F1-score show how well the model identifies minority class.

## 2. Train-Test Split 

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='TenYearCHD')
y = df['TenYearCHD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

train_medians = X_train.median() # fill missing values with training set medians
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (2966, 15)
Testing set shape: (1272, 15)


The stratified 70/30 split give 2966 training rows and 1272 test rows with similar class proportions. Setting **random_state=42** makes the split reproducible. Missing values in both sets are filled using medians calculated from training set, which prevents information from test set from affecting preprocessing.

## 3. Baseline Logistic Regression 

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)

# create function to avoid repetition
def evaluate_model(model, X_te, y_te, name = "Model Evaluation"):
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)

    print(f"--- {name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    tn, fp, fn, tp = cm.ravel()
    return [acc, prec, rec, f1, tn, fp, fn, tp]


base_model_lr = LogisticRegression(max_iter=5000, random_state=42)
base_model_lr.fit(X_train, y_train)

baseline_results = evaluate_model(base_model_lr, X_test, y_test, "Model A: Baseline Logistic Regression (Unscaled)")


--- Model A: Baseline Logistic Regression (Unscaled) ---
Accuracy:  0.8467
Precision: 0.4643
Recall:    0.0674
F1-Score:  0.1176
Confusion Matrix:
 [[1064   15]
 [ 180   13]]


The baseline uses all 15 input columns with their original scales and class distribution.

- **Accuracy** is the share of all test predictions that were correct.
- **Precision** is the share of predicted CHD cases that actually developed CHD.
- **Recall** is the share of actual CHD cases the model detected.
- **F1-score** is the balance mean of precision and recall.
- **Confusion matrix** show the counts of true negatives, false positives, false negatives and true positives. Rows are actual classes, and columns are predicted classes.

The baseline detected only **13 of 193** CHD cases despite achieving **84.67%** accuracy.

## 4. Effect of Feature Scaling

In [13]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
print("--- Numeric Columns ---")
print(numeric_cols)
print()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

scaled_model = LogisticRegression(max_iter=5000, random_state=42)
scaled_model.fit(X_train_scaled, y_train)

evaluate_model(base_model_lr, X_test, y_test, "Model A: Baseline Logistic Regression (Unscaled)")
print()
scaled_results = evaluate_model(scaled_model, X_test_scaled, y_test, "Model B: Scaled Logistic Regression")

--- Numeric Columns ---
['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

--- Model A: Baseline Logistic Regression (Unscaled) ---
Accuracy:  0.8467
Precision: 0.4643
Recall:    0.0674
F1-Score:  0.1176
Confusion Matrix:
 [[1064   15]
 [ 180   13]]

--- Model B: Scaled Logistic Regression ---
Accuracy:  0.8475
Precision: 0.4815
Recall:    0.0674
F1-Score:  0.1182
Confusion Matrix:
 [[1065   14]
 [ 180   13]]


The scaler learn the mean and standard deviation for each selected numerical feature from training set and applies same transformation to the test set. Putting numerical features on a similar scale helps the model train more easily and apply its penalty more evenly across features (Ref - https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html).

In our case, scaling slightly improved accuracy, precision and F1-score. Recall stayed at **6.74%**, and the scaled model made one fewer false-positive prediction.

## 5. Handling the Imbalanced Target Variable

In [14]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

All models in this section use scaled features and the same test set. Resampling is applied only to the training data.

- **Random undersampling** removes randomly selected majority-class rows.
- **Random oversampling** duplicate randomly selected minority-class rows.
- **SMOTE** creates new minority-class examples by interpolating between nearby minority-class examples.


(Ref - Imbalanced Learn Documentations)

In [15]:
# Model A: Original training data (already trained as scaled_model)
evaluate_model(scaled_model, X_test_scaled, y_test, "Model A: Original Data (Scaled)")

# Model B: Random UnderSampling
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled, y_train)

model_rus = LogisticRegression(max_iter=5000, random_state=42)
model_rus.fit(X_train_rus, y_train_rus)
under_results = evaluate_model(model_rus, X_test_scaled, y_test, "Model B: Random Undersampling (Scaled)")

# Model C: Random Oversampling
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_scaled, y_train)

model_ros = LogisticRegression(max_iter=5000, random_state=42)
model_ros.fit(X_train_ros, y_train_ros)
over_results = evaluate_model(model_ros, X_test_scaled, y_test, "Model C: Random Oversampling (Scaled)")

# Model D: SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

model_smote = LogisticRegression(max_iter=5000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)
smote_results = evaluate_model(model_smote, X_test_scaled, y_test, "Model D: SMOTE (Scaled)")

--- Model A: Original Data (Scaled) ---
Accuracy:  0.8475
Precision: 0.4815
Recall:    0.0674
F1-Score:  0.1182
Confusion Matrix:
 [[1065   14]
 [ 180   13]]
--- Model B: Random Undersampling (Scaled) ---
Accuracy:  0.6659
Precision: 0.2511
Recall:    0.6062
F1-Score:  0.3551
Confusion Matrix:
 [[730 349]
 [ 76 117]]
--- Model C: Random Oversampling (Scaled) ---
Accuracy:  0.6635
Precision: 0.2516
Recall:    0.6166
F1-Score:  0.3574
Confusion Matrix:
 [[725 354]
 [ 74 119]]
--- Model D: SMOTE (Scaled) ---
Accuracy:  0.6549
Precision: 0.2349
Recall:    0.5648
F1-Score:  0.3318
Confusion Matrix:
 [[724 355]
 [ 84 109]]


All three resampling methods increased CHD recall. 

Random oversampling detected **119 of 193** CHD cases, random undersampling detected **117**, and SMOTE detected **109**. The scaled model trained on original class distribution detected **13**. The gain in recall came with lower precision because the models also predicted more false positives.

## 6. Model Comparison

All five approaches were evaluated on the same 1,272 test observations. TN, FP, FN and TP are the four confusion matrix count.

In [16]:
# crate dataFrame for table setup
comparison = pd.DataFrame(
    [baseline_results, scaled_results, under_results, over_results, smote_results],
    index=["Original data (unscaled)", "Original data (scaled)",
        "Random undersampling", "Random oversampling", "SMOTE"],
    columns=["Accuracy", "Precision", "Recall", "F1-score", "TN", "FP", "FN", "TP"]
)

# add a new column into comparison table at fist col position
comparison.insert(0, "Feature scaling", ["No", "Yes", "Yes", "Yes", "Yes"])

display(comparison.round(4))

,Feature scaling,Accuracy,Precision,Recall,F1-score,TN,FP,FN,TP
Original data (unscaled),No,0.8467,0.4643,0.0674,0.1176,1064,15,180,13
Original data (scaled),Yes,0.8475,0.4815,0.0674,0.1182,1065,14,180,13
Random undersampling,Yes,0.6659,0.2511,0.6062,0.3551,730,349,76,117
Random oversampling,Yes,0.6635,0.2516,0.6166,0.3574,725,354,74,119
SMOTE,Yes,0.6549,0.2349,0.5648,0.3318,724,355,84,109


## 7. Final Conclusion

Scaling alone made little difference to the test results. Resampling had a much larger effect here in our case. SMOTE increased recall from **6.74%** to **56.48%** compared with the scaled model trained on the original data, but precision fell from **48.15%** to **23.49%**.

Random oversampling had the highest recall (**61.66%**) and F1-score (**0.3574**). It detected **119** of the 193 CHD cases and produced **354** false positives. 

Random undersampling performed similarly, detecting **117** cases with **349** false positives. The scaled model trained on the original data had the highest accuracy (**84.75%**) and precision (**48.15%**), but it missed **180** CHD cases. 

On this test set, random oversampling gave the best precision-recall balance by F1-score.